##  **LAB 06 – AUTOMATIC SPEECH RECOGNITION (ASR)**



In [ ]:
# ASR libraries
!pip install -q git+https://github.com/openai/whisper.git
!pip install -q transformers
!pip install -q datasets soundfile librosa jiwer evaluate
!pip install -q torchaudio
!pip install -q pydub ffmpeg-python
!pip install -q torchcodec
!apt install ffmpeg -y


import torch
import torchaudio
import librosa
from transformers import pipeline
from IPython.display import Audio, Javascript, display

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Using device: {device}")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 34.6 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.
 Using device: cuda


# ==============================================
#  PART 1 – USING PRE-TRAINED MODELS
# ==============================================

- Use audio from https://github.com/x4nth055/pythoncode-tutorials/raw/master/machine-learning/speech-recognition/30-4447-0004.wav

- Test Whisper model: whisper-base.en
- Test Wave2vec model: facebook/wav2vec2-base-960h"

**Download the sample audio file and save it somewhere**

**Play it**

In [3]:
# Download a sample audio file
# This audio file will be used to test the pre-trained ASR models.
audio_url = "https://github.com/x4nth055/pythoncode-tutorials/raw/master/machine-learning/speech-recognition/30-4447-0004.wav"
import os
os.makedirs("/content/audio", exist_ok=True)
!wget -q $audio_url -O /content/audio/sample.wav

# Play the audio



**Whisper**

Whisper is that there is a single model for almost all the world languages, and you only have to specify the language by prepending the decoding with special language tokens.

The below tables show the different Whisper models, their number of parameters, size, and links to multilingual and English-only models:

| Model  | Number of Parameters | Size       | Multilingual model        | English-only model      |
|--------|--------------------|------------|---------------------------|------------------------|
| Tiny   | 39M                | ~151MB     | openai/whisper-tiny       | openai/whisper-tiny.en |
| Base   | 74M                | ~290MB     | openai/whisper-base       | openai/whisper-base.en |
| Small  | 244M               | ~967MB     | openai/whisper-small      | openai/whisper-small.en|
| Medium | 769M               | ~3.06GB    | openai/whisper-medium     | openai/whisper-medium.en|
| Large  | 1550M              | ~6.17GB    | openai/whisper-large-v2   | N/A                    |


In [ ]:
# Load the pre-trained Whisper model and processor
#We use the WhisperProcessor to process our audio file, and WhisperForConditionalGeneration for loading the model.
from transformers import WhisperProcessor, WhisperForConditionalGeneration

model_name = "openai/whisper-base.en"
processor_whisper = WhisperProcessor.from_pretrained
model_whisper = WhisperForConditionalGeneration.from_pretrained

In [ ]:
# Load and preprocess audio
# Convert stereo/any sample rate to a single-channel 16kHz signal
speech, sr =
speech =


**Questions** Why do we resample to 16kHz for Whisper?

In [ ]:
# Convert audio to model input features
input_features = processor_whisper(speech, sampling_rate=16000, return_tensors="pt").input_features.to(device)
input_features.shape

torch.Size([1, 80, 3000])

In [ ]:
# special decoder tokens, to begin with during inference
# It is important when using a multilingual Whisper model to ensure that Whisper always decodes in the right language
forced_decoder_ids = processor_whisper.get_decoder_prompt_ids(language="english", task="transcribe")
forced_decoder_ids

[(1, 50258), (2, 50358), (3, 50362)]

In [ ]:
# perform inference
predicted_ids = model_whisper.generate
predicted_ids.shape

torch.Size([1, 56])

In [ ]:
# decode the IDs to text
transcription_whisper = processor_whisper.batch_decode
print("Whisper Transcription:\n", transcription_whisper)

Whisper Transcription:
  Three ladies almost always at the service of an invitation from Hartfield, and who were fetched and carried home so often that Mr. Woodhouse sought it no hardship for either James or the horses. Had it taken place only once a year, it would have been a grievance.


**Wave2Vec2**

There are two most used model architectures and weights for wav2vec2.
- wav2vec2-base-960h is a base architecture with about 360MB of size, it achieved a 3.4% Word Error Rate (WER) on the clean test set and was trained on 960 hours of LibriSpeech dataset on 16kHz sampled speech audio.

- wav2vec2-large-960h-lv60-self is a larger model with about 1.18GB in size (probably won't fit your laptop RAM) but achieved 1.9% WER (the lower, the better) on the clean test set. So this one is much better for recognition but heavier and takes more time for inference. Feel free to choose which one suits you best.

Wav2Vec2 was trained using Connectionist Temporal Classification (CTC), so that's why we're using the Wav2Vec2ForCTC class for loading the model.

In [ ]:
# Wav2Vec2 Pretrained Model
# Load the pre-trained Wav2Vec2 model and processor
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

wav2vec_name = "facebook/wav2vec2-base-960h"
processor_w2v =
model_w2v =



In [ ]:
# Load and preprocess the same audio file
# We need to make sure the input audio file to the model has the sample rate of 16000Hz because wav2vec2 is trained on that
speech, sr =
speech =


In [ ]:
# Prepare audio for Wav2Vec2 and run inference
#We specify the sampling_rate and pass "pt" to return_tensors argument to get PyTorch tensors in the results.
input_values =

#Let's pass the vector into our model now but because we are doing inference only, not training,
#do not track gradient in pytorch to save CPU
with torch.no_grad():
    logits =


In [ ]:
# Passing the logits to torch.argmax() to get the likely prediction:
predicted_ids =

#Decoding them back into text, we also lower the text, as it's in all caps
transcription_w2v =
print(" Wav2Vec2 Transcription:\n", transcription_w2v)

 Wav2Vec2 Transcription:
 AND MISSUS GODDARD THREE LADIES ALMOST ALWAYS AT THE SERVICE OF AN INVITATION FROM HARTFIELD AND WHO WERE FETCHED AND CARRIED HOME SO OFTEN THAT MISTER WOODHOUSE THOUGHT IT NO HARDSHIP FOR EITHER JAMES OR THE HORSES HAD IT TAKEN PLACE ONLY ONCE A YEAR IT WOULD HAVE BEEN A GRIEVANCE


**Wrapping all the codes**

Create functions to load audio and get Whisper and Wave2vec2 transcription

In [ ]:
def load_audio(audio_path):
  """Load the audio file & convert to 16,000 sampling rate"""


def get_transcription_wav2vec2(audio_path, model, processor):


def get_transcription_whisper(audio_path, model, processor, language="english", skip_special_tokens=True):


**Compute WRE for both model**

The true sentence is:
"and Mrs. Goddard, three ladies almost always at the service of an invitation from Hartfield, and who were fetched and carried home so often that Mr. Woodhouse thought it no hardship for either James or the horses. Had it taken place only once a year, it would have been a grievance"

In [ ]:
true_sentence='and Mrs. Goddard, three ladies almost always at the service of an invitation from Hartfield, and who were fetched and carried home so often that Mr. Woodhouse thought it no hardship for either James or the horses. Had it taken place only once a year, it would have been a grievance'

In [ ]:
import evaluate

wer = evaluate.load("wer")
audio_path = "/content/audio/sample.wav"

#Run Whisper and wave2vec2


wer_w2v = wer.compute(references=[true_sentence], predictions=[transcription_w2v])
wer_whisper = wer.compute(references=[true_sentence], predictions=[transcription_whisper])

print("WER Wav2Vec2:", wer_w2v)
print("WER Whisper:", wer_whisper)

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a lar

WER Wav2Vec2: 0.17647058823529413
WER Whisper: 0.11764705882352941


# ==============================================
# PART 2 – RECORD AND TEST YOUR OWN VOICE
# ==============================================

**Read a sentence of your choice and check the performance of both models**

In [ ]:
from IPython.display import Javascript
from google.colab import output
import io, base64
from pydub import AudioSegment
import os

RECORD_JS = """
var recorder, gumStream;
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = e => resolve(e.target.result.split(',')[1]);
  reader.readAsDataURL(blob);
});

async function recordAudio() {
  gumStream = await navigator.mediaDevices.getUserMedia({audio: true});
  recorder = new MediaRecorder(gumStream);
  let data = [];

  recorder.ondataavailable = event => data.push(event.data);
  recorder.start();

  const stopped = new Promise((resolve, reject) => {
    recorder.onstop = resolve;
    recorder.onerror = event => reject(event.name);
  });

  const recorded = await stopped;
  const blob = new Blob(data);
  const arrayBuffer = await blob.arrayBuffer();
  const base64data = await b2text(blob);
  gumStream.getTracks().forEach(track => track.stop());
  google.colab.kernel.invokeFunction('notebook.record', [base64data], {});
}

function startRecording() {
  recordAudio();
  console.log(' Recording... Speak now and click "Stop Recording" when done.');
}

function stopRecording() {
  if (recorder && recorder.state === "recording") {
    recorder.stop();
    console.log(' Recording stopped.');
  } else {
    console.log("No active recording to stop.");
  }
}

displayButtons();

function displayButtons() {
  const div = document.createElement('div');
  div.innerHTML = `
    <button onclick="startRecording()" style="font-size:16px;margin:5px;"> Start Recording</button>
    <button onclick="stopRecording()" style="font-size:16px;margin:5px;">⏹ Stop Recording</button>
  `;
  document.body.appendChild(div);
}
"""

recorded_audio = None

def record_audio(filename='/content/audio/my_sentence.wav'):
    def _record(b64data):
        audio_bytes = base64.b64decode(b64data)
        with open("/content/audio_temp.webm", "wb") as f:
            f.write(audio_bytes)
        audio = AudioSegment.from_file("/content/audio_temp.webm", format="webm")
        audio.export(filename, format="wav")
        print(f" Saved recording to {filename}")
    output.register_callback('notebook.record', _record)
    display(Javascript(RECORD_JS))

record_audio()

<IPython.core.display.Javascript object>

 Saved recording to /content/audio/my_sentence.wav


In [ ]:
my_sentence=

In [ ]:
# Play the audio
Audio("/content/audio/my_sentence.wav")

In [ ]:
audio_path = "/content/audio/my_sentence.wav"

#Run Whisper and wave2vec2

print("WER Wav2Vec2:", wer_w2v)
print("Transcripit Wav2Vec2:", transcription_w2v)

print("WER Whisper:", wer_whisper)
print("Transcripit Wav2Vec2:", transcription_whisper)


**Questions**

 - Try a longer sentence or a different accent. How does WER change?  
 - Try a multilingual Whisper model on a non-English sentence.  
 - Visualize the audio spectrogram. Can you identify word boundaries?  
 - Add noise to the audio. How robust are the models?